# Pengujian Model di Cloud: Prediksi Customer Churn

**Nama:** Fauzi Fadhlurrohman
Notebook ini menguji sistem machine learning yang berjalan di **Google Cloud (Compute Engine)**. Model pipeline TFX dilayani oleh **TensorFlow Serving**, dibungkus **gateway REST** yang menerima fitur pelanggan dalam JSON dan mengekspor metrik **Prometheus**.

- Web app: http://34.122.18.16:8080/
- Endpoint prediksi: `POST http://34.122.18.16:8080/predict`
- Metrik Prometheus: http://34.122.18.16:8080/metrics

In [1]:
import json

import pandas as pd
import requests

BASE_URL = "http://34.122.18.16:8080"
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

print(requests.get(BASE_URL + "/health", timeout=30).json())

{'status': 'ok'}


## 1. Prediksi satu pelanggan

Pelanggan dengan kontrak bulanan, internet *fiber optic*, dan masa berlangganan pendek (profil berisiko churn).

In [2]:
risky = {
    "gender": "Female", "SeniorCitizen": 0, "Partner": "No", "Dependents": "No",
    "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No",
    "StreamingMovies": "No", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 80.35, "TotalCharges": 160.7,
}
response = requests.post(BASE_URL + "/predict", json=risky, timeout=60)
print(response.status_code, response.json())

200 {'label': 'churn', 'probability': 0.736}


Pelanggan dengan kontrak dua tahun, banyak layanan tambahan, dan masa berlangganan panjang (profil aman).

In [3]:
safe = {
    "gender": "Male", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "Yes",
    "tenure": 60, "PhoneService": "Yes", "MultipleLines": "Yes",
    "InternetService": "DSL", "OnlineSecurity": "Yes", "OnlineBackup": "Yes",
    "DeviceProtection": "Yes", "TechSupport": "Yes", "StreamingTV": "No",
    "StreamingMovies": "No", "Contract": "Two year", "PaperlessBilling": "No",
    "PaymentMethod": "Credit card (automatic)", "MonthlyCharges": 60.0, "TotalCharges": 3600.0,
}
response = requests.post(BASE_URL + "/predict", json=safe, timeout=60)
print(response.status_code, response.json())

200 {'label': 'tidak_churn', 'probability': 0.0121}


## 2. Prediksi batch dan perbandingan dengan label asli

Sebanyak 200 pelanggan acak dikirim dalam satu request, lalu hasilnya dibandingkan dengan label aslinya.

In [4]:
df = pd.read_csv(DATA_URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
df = df.drop(columns=["customerID"])
sample = df.sample(200, random_state=7).reset_index(drop=True)

rows = sample.drop(columns=["Churn"]).to_dict(orient="records")
response = requests.post(BASE_URL + "/predict", json=rows, timeout=120)
predictions = response.json()["predictions"]
print(response.status_code, len(predictions), "prediksi")

sample["probabilitas_churn"] = [p["probability"] for p in predictions]
sample["prediksi"] = ["Yes" if p["label"] == "churn" else "No" for p in predictions]
print(sample[["tenure", "Contract", "MonthlyCharges", "Churn", "probabilitas_churn", "prediksi"]].head(8).to_string())

200 200 prediksi
   tenure        Contract  MonthlyCharges Churn  probabilitas_churn prediksi
0      34  Month-to-month           62.65   Yes              0.1163       No
1      18  Month-to-month           85.45   Yes              0.5081      Yes
2       9  Month-to-month           98.30   Yes              0.6516      Yes
3      56        One year          106.80    No              0.3399       No
4      42        One year          108.30    No              0.3287       No
5      25  Month-to-month           58.60   Yes              0.2796       No
6      33  Month-to-month           80.60   Yes              0.4242       No
7       2  Month-to-month           79.55    No              0.6772      Yes


In [5]:
actual = (sample["Churn"] == "Yes").astype(int)
predicted = (sample["prediksi"] == "Yes").astype(int)
tp = int(((predicted == 1) & (actual == 1)).sum())
fp = int(((predicted == 1) & (actual == 0)).sum())
fn = int(((predicted == 0) & (actual == 1)).sum())
print(f"Accuracy : {(predicted == actual).mean():.4f}")
print(f"Precision: {tp / max(tp + fp, 1):.4f}")
print(f"Recall   : {tp / max(tp + fn, 1):.4f}")

Accuracy : 0.7900
Precision: 0.6596
Recall   : 0.5439


## 3. Validasi input

Gateway menolak data yang tidak valid dengan status 422; error ini tercatat pada metrik `churn_errors_total` dan `churn_http_requests_total{status="422"}` di Prometheus.

In [6]:
invalid = dict(risky, Contract="Lifetime")
response = requests.post(BASE_URL + "/predict", json=invalid, timeout=60)
print(response.status_code, response.json())

422 {'error': "nilai Contract harus salah satu dari ['Month-to-month', 'One year', 'Two year']"}


## 4. Metrik Prometheus

Endpoint `/metrics` mengekspor metrik yang di-scrape Prometheus setiap 15 detik. Cuplikan metrik utama:

In [7]:
metrics = requests.get(BASE_URL + "/metrics", timeout=30).text
for line in metrics.splitlines():
    if line.startswith(("churn_predictions_total", "churn_model_available", "churn_errors_total", "churn_http_requests_total")):
        print(line)

churn_http_requests_total{endpoint="/metrics",method="GET",status="200"} 2.0
churn_http_requests_total{endpoint="/health",method="GET",status="200"} 2.0
churn_http_requests_total{endpoint="/v1/models",method="GET",status="200"} 2.0
churn_http_requests_total{endpoint="/predict",method="POST",status="200"} 3.0
churn_http_requests_total{endpoint="/predict",method="POST",status="422"} 1.0
churn_predictions_total{label="churn"} 48.0
churn_predictions_total{label="tidak_churn"} 154.0
churn_errors_total{type="validation"} 1.0
churn_model_available 1.0


## 5. Kesimpulan

Sistem berjalan di cloud: endpoint publik menjawab seluruh request dan mengembalikan probabilitas churn dari model TFX. Hasil pada 200 contoh konsisten dengan evaluasi pipeline (accuracy sekitar 0,80). Hasil pemantauan Prometheus ada di `README.md` dan folder `monitoring/`.